# Roll No: 2025AG05637 | DM4ML Lab Assignment

## DM4ML Lab Assignment 1 — Kaggle API Download & pandas Preprocessing for Apache Druid

**Course:** Data Management for Machine Learning (DM4ML)
**Datasource (Druid):** `dm4ml_2025AG05637`

**Dataset:** *Global Superstore* — Kaggle `fatihilhan/global-superstore-dataset` (~51,290 order lines, 2011–2014)

| Requirement | This dataset |
|---|---|
| Rows 10k–500k | ~51k order lines |
| ≥3–4 dimensions | time (Order Date), Market, Region, Country, Segment, Category, Sub-Category, Ship Mode, Order Priority |
| ≥2–3 numeric metrics | Sales, Quantity, Discount, Profit, Shipping Cost |

**Notebook flow**
1. Step 1 — Authenticate with the Kaggle API and download the dataset programmatically, load it into pandas.
2. Step 2 — Eight pandas preprocessing steps.
3. Output — Export a Druid-compatible **newline-delimited JSON** file (used for ingestion) plus a CSV copy.


## Step 1 — Download the dataset using the Kaggle API

### 1.1 Install / import libraries

In [1]:
# Install once if needed (uncomment):
# %pip install -q kaggle pandas

import os, json, glob, getpass, pathlib, datetime
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

pandas 3.1.0


### 1.2 Configure Kaggle API credentials

Per the official Kaggle API documentation, the CLI/library reads the API token (generated at **Kaggle → Settings → API → Create New Token**) from either:

* the environment variable `KAGGLE_API_TOKEN`, or
* the file `~/.kaggle/access_token` (permission `600`), or
* the legacy `~/.kaggle/kaggle.json` (`{"username": ..., "key": ...}`).

The token is **never written into this notebook**. If none of the above is configured, the cell below prompts for it (hidden input) and keeps it only in this kernel's memory.

In [2]:
KAGGLE_DIR = pathlib.Path.home() / ".kaggle"
token_file = KAGGLE_DIR / "access_token"
legacy_file = KAGGLE_DIR / "kaggle.json"

configured = bool(os.environ.get("KAGGLE_API_TOKEN")) or token_file.exists() or legacy_file.exists()
if not configured:
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Paste your Kaggle API token: ").strip()

print("KAGGLE_API_TOKEN env var set :", bool(os.environ.get("KAGGLE_API_TOKEN")))
print("~/.kaggle/access_token exists:", token_file.exists())
print("~/.kaggle/kaggle.json exists :", legacy_file.exists())

KAGGLE_API_TOKEN env var set : False
~/.kaggle/access_token exists: True
~/.kaggle/kaggle.json exists : False


### 1.3 Authenticate and download programmatically

In [3]:
from kaggle.api.kaggle_api_extended import KaggleApi

DATASET  = "fatihilhan/global-superstore-dataset"
RAW_DIR  = pathlib.Path("data/raw")
OUT_DIR  = pathlib.Path("data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()                      # reads the token configured above
print("Authenticated with Kaggle API")

# List the files in the dataset, then download + unzip them
files = api.dataset_list_files(DATASET).files
for f in files:
    print(" -", f.name, getattr(f, "total_bytes", ""), "bytes")

api.dataset_download_files(DATASET, path=str(RAW_DIR), unzip=True, quiet=False)
print("Downloaded to:", RAW_DIR.resolve())
print(sorted(p.name for p in RAW_DIR.iterdir()))

Authenticated with Kaggle API
 - superstore.csv 15844267 bytes
Dataset URL: https://www.kaggle.com/datasets/fatihilhan/global-superstore-dataset


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3.19M/3.19M [00:08<00:00, 403kB/s]



Downloaded to: /mnt/c/DM4ML_Druid_Lab/data/raw
['superstore.csv']


### 1.4 Load the downloaded file into a pandas DataFrame

In [4]:
candidates = sorted(glob.glob(str(RAW_DIR / "**" / "*.csv"), recursive=True)) \
           + sorted(glob.glob(str(RAW_DIR / "**" / "*.xls*"), recursive=True))
assert candidates, "No CSV/Excel file found in data/raw"
src = max(candidates, key=os.path.getsize)          # main data file
print("Loading:", src)

if src.endswith(".csv"):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            raw = pd.read_csv(src, encoding=enc, low_memory=False)
            break
        except UnicodeDecodeError:
            continue
else:
    raw = pd.read_excel(src)

print("Raw shape:", raw.shape)
raw.head()

Loading: data/raw/superstore.csv
Raw shape: (51290, 27)


,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


In [5]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        51290 non-null  str    
 1   City            51290 non-null  str    
 2   Country         51290 non-null  str    
 3   Customer.ID     51290 non-null  str    
 4   Customer.Name   51290 non-null  str    
 5   Discount        51290 non-null  float64
 6   Market          51290 non-null  str    
 7   记录数             51290 non-null  int64  
 8   Order.Date      51290 non-null  str    
 9   Order.ID        51290 non-null  str    
 10  Order.Priority  51290 non-null  str    
 11  Product.ID      51290 non-null  str    
 12  Product.Name    51290 non-null  str    
 13  Profit          51290 non-null  float64
 14  Quantity        51290 non-null  int64  
 15  Region          51290 non-null  str    
 16  Row.ID          51290 non-null  int64  
 17  Sales           51290 non-null  int64  
 1

In [6]:
raw.describe(include="all").T.head(30)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Category,51290,3,Office Supplies,31273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,51290,3636,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,51290,147,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer.ID,51290,4873,JG-158051,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer.Name,51290,795,Muhammed Yedwab,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Discount,51290.0,NaN,NaN,NaN,0.142908,0.21228,0.0,0.0,0.0,0.2,0.85
Market,51290,7,APAC,11002,NaN,NaN,NaN,NaN,NaN,NaN,NaN
记录数,51290.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
Order.Date,51290,1430,2014-06-18 00:00:00.000,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order.ID,51290,25035,CA-2014-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 — Data cleaning & preprocessing (pandas)

Eight preprocessing steps are applied (the assignment requires at least five). A log records the row count after each step.

In [7]:
df = raw.copy()
log = []
def record(step, note=""):
    log.append({"step": step, "rows": len(df), "note": note})
    print(f"{step:<45} rows={len(df):>7,}  {note}")
record("0. Raw data loaded")

0. Raw data loaded                            rows= 51,290  


### 2.1 Standardise column names and drop irrelevant columns
Kaggle headers such as `Order.Date`, `Sub.Category` are converted to `snake_case` (`order_date`, `sub_category`). The constant helper column `记录数` ("record count", always 1) and the surrogate key `Row.ID` carry no analytical information and are dropped; the source `Year`/`weeknum` columns are dropped because they are re-derived from the cleaned date in step 2.6.

In [8]:
import re
def snake(c):
    c = re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_")
    return re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c).lower() if c else c

drop_cols = [c for c in df.columns if not re.search(r"[A-Za-z]", str(c))]   # e.g. '记录数'
df = df.drop(columns=drop_cols)
df.columns = [snake(c) for c in df.columns]
df = df.drop(columns=[c for c in ["row_id", "year", "weeknum"] if c in df.columns])

required = ["order_id", "order_date", "ship_date", "ship_mode", "segment", "market", "region",
            "country", "category", "sub_category", "sales", "quantity", "discount", "profit",
            "shipping_cost", "order_priority"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"Expected columns not found: {missing_cols}"
print("Dropped:", drop_cols + ["row_id", "year", "weeknum"])
print("Columns:", list(df.columns))
record("1. Standardised names / dropped columns")

Dropped: ['记录数', 'row_id', 'year', 'weeknum']
Columns: ['category', 'city', 'country', 'customer_id', 'customer_name', 'discount', 'market', 'order_date', 'order_id', 'order_priority', 'product_id', 'product_name', 'profit', 'quantity', 'region', 'sales', 'segment', 'ship_date', 'ship_mode', 'shipping_cost', 'state', 'sub_category', 'market2']
1. Standardised names / dropped columns       rows= 51,290  


### 2.2 Trim whitespace in text columns and normalise empty strings to NaN

In [9]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df[c] = df[c].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "NULL": pd.NA})
record("2. Trimmed text columns")

2. Trimmed text columns                       rows= 51,290  


### 2.3 Convert data types
* `order_date`, `ship_date` → `datetime64` (ISO parse first; falls back to day-first parsing if the file uses `dd-mm-yyyy`).
* Numeric metrics → numeric (`coerce` turns stray text into NaN, handled in 2.4).
* Low-cardinality dimensions → `category` dtype (smaller memory, faster group-bys).

In [10]:
def parse_dates(s):
    d = pd.to_datetime(s, format="ISO8601", errors="coerce")
    if d.isna().mean() > 0.01:
        d = pd.to_datetime(s, format="mixed", dayfirst=True, errors="coerce")
    return d

df["order_date"] = parse_dates(df["order_date"])
df["ship_date"]  = parse_dates(df["ship_date"])

num_cols = ["sales", "quantity", "discount", "profit", "shipping_cost"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

cat_cols = ["ship_mode", "segment", "market", "region", "country", "category",
            "sub_category", "order_priority"]
for c in cat_cols:
    df[c] = df[c].astype("category")

print(df.dtypes)
record("3. Converted data types")

category                category
city                      string
country                 category
customer_id               string
customer_name             string
discount                 float64
market                  category
order_date        datetime64[us]
order_id                  string
order_priority          category
product_id                string
product_name              string
profit                   float64
quantity                   int64
region                  category
sales                      int64
segment                 category
ship_date         datetime64[us]
ship_mode               category
shipping_cost            float64
state                     string
sub_category            category
market2                   string
dtype: object
3. Converted data types                       rows= 51,290  


### 2.4 Handle missing values
* Rows missing a **critical** field (dates, sales, quantity, category, market) cannot be ingested meaningfully → dropped.
* `discount` / `shipping_cost` missing → filled with 0 (no discount / no shipping charge recorded).
* `profit` missing → filled with the median profit margin of the same sub-category × sales.
* Remaining categorical gaps → `"Unknown"`.

In [11]:
print("Missing values before:\n", df.isna().sum()[df.isna().sum() > 0])

critical = ["order_date", "ship_date", "sales", "quantity", "category", "market"]
before = len(df)
df = df.dropna(subset=critical)

df["discount"] = df["discount"].fillna(0)
df["shipping_cost"] = df["shipping_cost"].fillna(0)

margin = (df["profit"] / df["sales"]).groupby(df["sub_category"], observed=True).transform("median")
df["profit"] = df["profit"].fillna((margin * df["sales"]).round(4))

for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].cat.add_categories("Unknown").fillna("Unknown")
for c in ["city", "state", "customer_name", "product_name"]:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

print("Missing values after:", int(df[critical + num_cols + cat_cols].isna().sum().sum()))
record("4. Handled missing values", f"dropped {before - len(df)} rows")

Missing values before:
 Series([], dtype: int64)
Missing values after: 0
4. Handled missing values                     rows= 51,290  dropped 0 rows


### 2.5 Remove duplicate records
Exact duplicate rows (same order, product, date and amounts after dropping `row_id`) are removed.

In [12]:
before = len(df)
dups = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
record("5. Removed duplicates", f"removed {before - len(df)} (found {dups})")

5. Removed duplicates                         rows= 51,290  removed 0 (found 0)


### 2.6 Feature engineering — date parts and shipping time
New columns: `order_year`, `order_quarter` (`Q1`–`Q4`), `order_month` (`01`–`12`), `order_month_name`, `order_day_of_week`, and `ship_days` = ship date − order date.

In [13]:
df["order_year"]        = df["order_date"].dt.year.astype(str)
df["order_quarter"]     = "Q" + df["order_date"].dt.quarter.astype(str)
df["order_month"]       = df["order_date"].dt.month.map("{:02d}".format)
df["order_month_name"]  = df["order_date"].dt.strftime("%b")
df["order_day_of_week"] = df["order_date"].dt.day_name()
df["ship_days"]         = (df["ship_date"] - df["order_date"]).dt.days
record("6. Engineered date features")
df[["order_date", "order_year", "order_quarter", "order_month", "order_day_of_week", "ship_days"]].head()

6. Engineered date features                   rows= 51,290  


,order_date,order_year,order_quarter,order_month,order_day_of_week,ship_days
0,2011-01-07,2011,Q1,01,Friday,2
1,2011-01-21,2011,Q1,01,Friday,5
2,2011-08-05,2011,Q3,08,Friday,4
3,2011-08-05,2011,Q3,08,Friday,4
4,2011-09-29,2011,Q3,09,Thursday,4


### 2.7 Derived metric columns
* `unit_price` = sales ÷ quantity
* `gross_sales` = list value before discount = sales ÷ (1 − discount)
* `discount_amount` = gross_sales − sales
* `cost` = sales − profit
* `profit_margin` = profit ÷ sales

In [14]:
df["unit_price"]      = (df["sales"] / df["quantity"]).round(4)
df["gross_sales"]     = (df["sales"] / (1 - df["discount"].clip(upper=0.99))).round(4)
df["discount_amount"] = (df["gross_sales"] - df["sales"]).round(4)
df["cost"]            = (df["sales"] - df["profit"]).round(4)
df["profit_margin"]   = (df["profit"] / df["sales"]).round(4)
record("7. Created derived metrics")
df[["sales", "quantity", "discount", "unit_price", "gross_sales", "discount_amount", "cost", "profit_margin"]].describe().T

7. Created derived metrics                    rows= 51,290  


/home/abhinand_ntk/dm4ml-venv/lib/python3.14/site-packages/pandas/core/nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2


,count,mean,std,min,25%,50%,75%,max
sales,51290.0,246.498440,487.567175,0.0000,31.0000,85.00000,251.000000,22638.0000
quantity,51290.0,3.476545,2.278766,1.0000,2.0000,3.00000,5.000000,14.0000
discount,51290.0,0.142908,0.212280,0.0000,0.0000,0.00000,0.200000,0.8500
unit_price,51290.0,71.659964,113.581076,0.0000,12.0000,29.50000,82.000000,3773.0000
gross_sales,51290.0,292.589759,600.616002,0.0000,38.3333,101.00000,296.000000,45276.0000
discount_amount,51290.0,46.091319,209.261790,0.0000,0.0000,0.00000,25.500000,22638.0000
cost,51290.0,217.887458,430.902252,0.4172,26.9200,73.65000,223.077625,24449.0784
profit_margin,51290.0,-inf,NaN,-inf,0.0000,0.16925,0.333100,0.5828


### 2.8 Filter invalid records and outliers
* Invalid business records: `quantity ≤ 0`, `sales ≤ 0`, `discount` outside [0, 1), `ship_date` earlier than `order_date`.
* Outliers: shipping time above 30 days, and lines whose `unit_price` lies beyond **Q3 + 3×IQR** *within their own sub-category* (prices are compared like-for-like, so expensive product groups such as copiers are not removed wholesale).

In [15]:
before = len(df)
valid = (df["quantity"] > 0) & (df["sales"] > 0) & df["discount"].between(0, 0.99) & (df["ship_days"].between(0, 30))
df = df[valid]
n_invalid = before - len(df)

g = df.groupby("sub_category", observed=True)["unit_price"]
q1, q3 = g.transform(lambda s: s.quantile(0.25)), g.transform(lambda s: s.quantile(0.75))
upper = q3 + 3 * (q3 - q1)
n_out = int((df["unit_price"] > upper).sum())
df = df[df["unit_price"] <= upper].reset_index(drop=True)
record("8. Filtered invalid records & outliers", f"invalid={n_invalid}, price outliers={n_out}")

8. Filtered invalid records & outliers        rows= 50,425  invalid=1, price outliers=864


### Preprocessing summary

In [16]:
summary = pd.DataFrame(log)
summary["rows_removed"] = (-summary["rows"].diff()).fillna(0).astype(int)
display(summary)
print("Final shape:", df.shape)
print("Date range :", df["order_date"].min().date(), "→", df["order_date"].max().date())
df.groupby("category", observed=True)[["sales", "profit", "quantity"]].sum().round(2)

,step,rows,note,rows_removed
0,0. Raw data loaded,51290,,0
1,1. Standardised names / dropped columns,51290,,0
2,2. Trimmed text columns,51290,,0
3,3. Converted data types,51290,,0
4,4. Handled missing values,51290,dropped 0 rows,0
5,5. Removed duplicates,51290,removed 0 (found 0),0
6,6. Engineered date features,51290,,0
7,7. Created derived metrics,51290,,0
8,8. Filtered invalid records & outliers,50425,"invalid=1, price outliers=864",865


Final shape: (50425, 34)
Date range : 2011-01-01 → 2014-12-31


,sales,profit,quantity
category,,,
Furniture,3873231,238083.15,34457
Office Supplies,3561331,484546.44,107486
Technology,3797228,450497.51,33355


## Output — export a Druid-compatible file

Druid's native batch ingestion reads **newline-delimited JSON** (one JSON object per line) via `inputFormat: {"type": "json"}`. This is the file referenced in `ingestion_spec.json`. A CSV copy is also written.

* `order_date` is written as `yyyy-MM-dd` — this becomes Druid's primary timestamp `__time`.
* Category columns are written as plain strings; metrics as numbers.

In [17]:
export_cols = ["order_date", "ship_date", "order_id", "market", "region", "country", "state", "city",
               "segment", "category", "sub_category", "ship_mode", "order_priority",
               "order_year", "order_quarter", "order_month", "order_month_name", "order_day_of_week",
               "sales", "quantity", "discount", "profit", "shipping_cost",
               "unit_price", "gross_sales", "discount_amount", "cost", "profit_margin", "ship_days"]
export_cols = [c for c in export_cols if c in df.columns]

out = df[export_cols].copy()
out["order_date"] = out["order_date"].dt.strftime("%Y-%m-%d")
out["ship_date"]  = out["ship_date"].dt.strftime("%Y-%m-%d")
for c in out.select_dtypes(include=["category", "string"]).columns:
    out[c] = out[c].astype(str)
out["quantity"]  = out["quantity"].astype(int)
out["ship_days"] = out["ship_days"].astype(int)

OUT_DIR.mkdir(exist_ok=True)
json_path = OUT_DIR / "superstore_clean.json"
csv_path  = OUT_DIR / "superstore_clean.csv"
out.to_json(json_path, orient="records", lines=True, force_ascii=False)
out.to_csv(csv_path, index=False)
print("Wrote", json_path.resolve(), f"({json_path.stat().st_size/1e6:.1f} MB)")
print("Wrote", csv_path.resolve(),  f"({csv_path.stat().st_size/1e6:.1f} MB)")

Wrote /mnt/c/DM4ML_Druid_Lab/data/superstore_clean.json (30.7 MB)
Wrote /mnt/c/DM4ML_Druid_Lab/data/superstore_clean.csv (10.5 MB)


### Validate the JSON output (every line parses, row count matches)

In [18]:
with open(json_path, encoding="utf-8") as f:
    lines = f.readlines()
parsed = [json.loads(l) for l in lines]             # raises if any line is invalid JSON
assert len(parsed) == len(out), "row count mismatch"
check = pd.read_json(json_path, lines=True)
print("Valid NDJSON lines:", len(parsed), "| columns:", len(check.columns))
print(json.dumps(parsed[0], indent=2, ensure_ascii=False))

Valid NDJSON lines: 50425 | columns: 29
{
  "order_date": "2011-01-07",
  "ship_date": "2011-01-09",
  "order_id": "CA-2011-130813",
  "market": "US",
  "region": "West",
  "country": "United States",
  "state": "California",
  "city": "Los Angeles",
  "segment": "Consumer",
  "category": "Office Supplies",
  "sub_category": "Paper",
  "ship_mode": "Second Class",
  "order_priority": "High",
  "order_year": "2011",
  "order_quarter": "Q1",
  "order_month": "01",
  "order_month_name": "Jan",
  "order_day_of_week": "Friday",
  "sales": 19,
  "quantity": 3,
  "discount": 0.0,
  "profit": 9.3312,
  "shipping_cost": 4.37,
  "unit_price": 6.3333,
  "gross_sales": 19.0,
  "discount_amount": 0.0,
  "cost": 9.6688,
  "profit_margin": 0.4911,
  "ship_days": 2
}


### Save a preprocessing summary (used in the report)

In [19]:
report_summary = {
    "dataset": DATASET,
    "source_file": os.path.basename(src),
    "raw_rows": int(raw.shape[0]), "raw_columns": int(raw.shape[1]),
    "clean_rows": int(out.shape[0]), "clean_columns": int(out.shape[1]),
    "date_min": str(df["order_date"].min().date()), "date_max": str(df["order_date"].max().date()),
    "steps": log,
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
}
pathlib.Path("results").mkdir(exist_ok=True)
with open("results/preprocessing_summary.json", "w") as f:
    json.dump(report_summary, f, indent=2)
print(json.dumps({k: v for k, v in report_summary.items() if k != "steps"}, indent=2))

{
  "dataset": "fatihilhan/global-superstore-dataset",
  "source_file": "superstore.csv",
  "raw_rows": 51290,
  "raw_columns": 27,
  "clean_rows": 50425,
  "clean_columns": 29,
  "date_min": "2011-01-01",
  "date_max": "2014-12-31",
  "generated_at": "2026-10-27T16:49:10"
}


## Next step
The NDJSON file `data/superstore_clean.json` is ingested into Apache Druid with `ingestion_spec.json` (datasource `dm4ml_2025AG05637`). See `RUN_GUIDE.md`.

*Roll No: 2025AG05637 | DM4ML Lab Assignment*